# 01. Environment Check

Run this on the HPC node BEFORE running any benchmarks. Confirms:
- Both 2080 Tis are visible to PyTorch
- CUDA round-trip works
- Diffusers loads SD 1.5 successfully
- xformers imports without errors
- nsys is available on PATH

If any cell fails, fix it before moving on.

In [ ]:
import subprocess

for cmd in [['nvidia-smi'], ['nvidia-smi', 'nvlink', '--status'], ['nvcc', '--version']]:
    print('$', ' '.join(cmd))
    out = subprocess.run(cmd, capture_output=True, text=True)
    print(out.stdout)
    print('-' * 60)

In [ ]:
import torch

print(f'torch version:       {torch.__version__}')
print(f'CUDA available:      {torch.cuda.is_available()}')
print(f'CUDA version (PT):   {torch.version.cuda}')
print(f'Device count:        {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    print(f'  [{i}] {torch.cuda.get_device_name(i)} - cap {torch.cuda.get_device_capability(i)}')

In [ ]:
# CUDA round-trip
x = torch.randn(1024, 1024, device='cuda:0')
y = x @ x.T
torch.cuda.synchronize()
print(f'GPU 0 matmul OK, sum={y.sum().item():.2f}')

if torch.cuda.device_count() >= 2:
    x2 = x.to('cuda:1')
    y2 = x2 @ x2.T
    torch.cuda.synchronize()
    print(f'GPU 1 matmul OK, sum={y2.sum().item():.2f}')

In [ ]:
# Tensor Core check - matmul in FP16 should hit Tensor Cores on Turing
import time

for dtype in [torch.float32, torch.float16, torch.bfloat16]:
    a = torch.randn(4096, 4096, dtype=dtype, device='cuda:0')
    b = torch.randn(4096, 4096, dtype=dtype, device='cuda:0')
    # Warmup
    for _ in range(3):
        _ = a @ b
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(10):
        _ = a @ b
    torch.cuda.synchronize()
    print(f'{str(dtype):20s}  {(time.perf_counter() - t0) / 10 * 1000:.2f} ms / matmul')

In [ ]:
# Diffusers + SD 1.5 load test
from diffusers import StableDiffusionPipeline

pipe = StableDiffusionPipeline.from_pretrained(
    'stable-diffusion-v1-5/stable-diffusion-v1-5',
    torch_dtype=torch.float16,
    safety_checker=None,
    requires_safety_checker=False,
)
pipe = pipe.to('cuda:0')
print('SD 1.5 loaded, generating one quick test image...')
image = pipe('a red apple on a wooden table', num_inference_steps=15).images[0]
print('OK')
image

In [ ]:
# xformers import test
try:
    import xformers
    print(f'xformers version: {xformers.__version__}')
    pipe.enable_xformers_memory_efficient_attention()
    print('xformers attention enabled OK')
except Exception as e:
    print(f'xformers failed: {e}')
    print('NOTE: xformers depends on the exact PyTorch version. If it fails, pin pytorch=2.4.0')

In [ ]:
# nsys availability
out = subprocess.run(['nsys', '--version'], capture_output=True, text=True)
print(out.stdout or out.stderr)